In [1]:
import itertools
import random
import csv

In [2]:
def write_csv_dict(filename, data):
    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)

In [ ]:
class AgreementTemplateGenerator:
    def __init__(self):
        self.subjects = {
            'sg': '[wug]',
            'pl': '[wugs]'
        }
        
        self.verbs = [
            # Auxiliaries / Copulas
            {'sg': 'is', 'pl': 'are', 'comp': 'happy'},
            {'sg': 'was', 'pl': 'were', 'comp': 'sleeping'},
            {'sg': 'has', 'pl': 'have', 'comp': 'arrived'},
            {'sg': 'does', 'pl': 'do', 'comp': 'understand'},
            
            # Lexical Verbs
            {'sg': 'jumps', 'pl': 'jump', 'comp': 'high'},
            {'sg': 'opens', 'pl': 'open', 'comp': 'doors'}, 
            {'sg': 'runs', 'pl': 'run', 'comp': 'fast'},    
            {'sg': 'eats', 'pl': 'eat', 'comp': 'apples'},
            {'sg': 'sleeps', 'pl': 'sleep', 'comp': 'soundly'},
            {'sg': 'sings', 'pl': 'sing', 'comp': 'loudly'},
            {'sg': 'reads', 'pl': 'read', 'comp': 'books'},
            {'sg': 'writes', 'pl': 'write', 'comp': 'stories'},
            {'sg': 'thinks', 'pl': 'think', 'comp': 'deeply'},
            {'sg': 'swims', 'pl': 'swim', 'comp': 'well'},
            # {'sg': 'drives', 'pl': 'drive', 'comp': 'safely'},
            {'sg': 'flies', 'pl': 'fly', 'comp': 'high'},
            {'sg': 'dances', 'pl': 'dance', 'comp': 'gracefully'},
            # {'sg': 'paints', 'pl': 'paint', 'comp': 'beautifully'},
            # {'sg': 'builds', 'pl': 'build', 'comp': 'strongly'},
            # {'sg': 'sells', 'pl': 'sell', 'comp': 'quickly'},
            # {'sg': 'teaches', 'pl': 'teach', 'comp': 'well'},
            # {'sg': 'learns', 'pl': 'learn', 'comp': 'easily'},
            {'sg': 'plays', 'pl': 'play', 'comp': 'often'},
            {'sg': 'works', 'pl': 'work', 'comp': 'hard'},
            {'sg': 'listens', 'pl': 'listen', 'comp': 'carefully'},
            {'sg': 'watches', 'pl': 'watch', 'comp': 'closely'},
            {'sg': 'waits', 'pl': 'wait', 'comp': 'patiently'},
            {'sg': 'helps', 'pl': 'help', 'comp': 'frequently'},
            {'sg': 'calls', 'pl': 'call', 'comp': 'often'},
        ]
        
        self.distractors = [
            {'sg': 'tree', 'pl': 'trees'},
            {'sg': 'car', 'pl': 'cars'},
            {'sg': 'box', 'pl': 'boxes'},
            {'sg': 'cat', 'pl': 'cats'},
            {'sg': 'dog', 'pl': 'dogs'},
            {'sg': 'horse', 'pl': 'horses'},
            {'sg': 'woman', 'pl': 'women'},
            {'sg': 'man', 'pl': 'men'},
            {'sg': 'fox', 'pl': 'foxes'},
            {'sg': 'bird', 'pl': 'birds'},
        ]
        
        self.prepositions = [
            "near the",
            "behind the",
            "under the",
            "beside the",
            "in front of the",
            "next to the",
            "past the"
        ]

    def format_sentence(self, subj, prep_phrase, verb, comp):
        """Builds the sentence uniformly for all verb types."""
        prep_str = f" {prep_phrase}" if prep_phrase else ""
        comp_str = f" {comp}" if comp else ""
        
        raw_sentence = f"The {subj}{prep_str} {verb}{comp_str}"
        return f"{raw_sentence.strip()}."

    def build_prep_phrase(self, prep_parts):
        """Helper to format list of prepositional phrases grammatically."""
        if not prep_parts:
            return ""
        elif len(prep_parts) == 1:
            return prep_parts[0]
        elif len(prep_parts) == 2:
            return f"{prep_parts[0]} and {prep_parts[1]}"
        else:
            return ", ".join(prep_parts[:-1]) + f", and {prep_parts[-1]}"

    def generate_templates(self, max_distractors=4):
        items = []
        
        for verb_dict in self.verbs:
            comp = verb_dict['comp']
            
            for i in range(len(self.distractors)):
                for num_distractors in range(max_distractors + 1):
                    
                    current_distractors = [
                        self.distractors[(i + j) % len(self.distractors)] 
                        for j in range(num_distractors)
                    ]
                    
                    for subj_num in ['sg', 'pl']:
                        subj = self.subjects[subj_num]
                        good_verb = verb_dict[subj_num]
                        bad_verb = verb_dict['pl' if subj_num == 'sg' else 'sg']
                        
                        # --- NEW LOGIC: Force distractors to be the opposite number ---
                        target_dist_num = 'pl' if subj_num == 'sg' else 'sg'
                        
                        # Create a list where every distractor gets this opposite number
                        d_nums = [target_dist_num] * num_distractors
                        
                        # 1. Build the condition string
                        condition_parts = [f"subj_{subj_num}"]
                        for j, d_num in enumerate(d_nums):
                            condition_parts.append(f"d{j+1}_{d_num}")
                        condition = "_".join(condition_parts)
                        
                        # 2. Build the prepositional phrase
                        prep_parts = [] 
                        for j, d_num in enumerate(d_nums):
                            prep = self.prepositions[(i + j) % len(self.prepositions)]
                            noun = current_distractors[j][d_num]
                            prep_parts.append(f"{prep} {noun}")
                            
                        prep_phrase = self.build_prep_phrase(prep_parts)
                        
                        # 3. Assemble and store
                        items.append({
                            'distractors': num_distractors,
                            'condition': condition,
                            'good': self.format_sentence(subj, prep_phrase, good_verb, comp),
                            'bad': self.format_sentence(subj, prep_phrase, bad_verb, comp)
                        })
                            
        return items

In [13]:
# Generate dataset
generator = AgreementTemplateGenerator()
dataset = generator.generate_templates()

In [14]:
len(dataset)

2300

In [15]:
four = [item for item in dataset if item['distractors'] == 4]

In [16]:
len(four)

460

In [17]:
four[:4]

[{'distractors': 4,
  'condition': 'subj_sg_d1_pl_d2_pl_d3_pl_d4_pl',
  'good': 'The [wug] near the trees, behind the cars, under the boxes, and beside the cats is happy.',
  'bad': 'The [wug] near the trees, behind the cars, under the boxes, and beside the cats are happy.'},
 {'distractors': 4,
  'condition': 'subj_pl_d1_sg_d2_sg_d3_sg_d4_sg',
  'good': 'The [wugs] near the tree, behind the car, under the box, and beside the cat are happy.',
  'bad': 'The [wugs] near the tree, behind the car, under the box, and beside the cat is happy.'},
 {'distractors': 4,
  'condition': 'subj_sg_d1_pl_d2_pl_d3_pl_d4_pl',
  'good': 'The [wug] behind the cars, under the boxes, beside the cats, and in front of the dogs is happy.',
  'bad': 'The [wug] behind the cars, under the boxes, beside the cats, and in front of the dogs are happy.'},
 {'distractors': 4,
  'condition': 'subj_pl_d1_sg_d2_sg_d3_sg_d4_sg',
  'good': 'The [wugs] behind the car, under the box, beside the cat, and in front of the dog ar

In [18]:
# add idx for each item
for idx, item in enumerate(dataset):
    item['idx'] = idx

In [19]:
# format so that idx is first column
dataset = [{k: item[k] for k in ['idx', 'distractors', 'condition', 'good', 'bad']} for item in dataset]

In [20]:
write_csv_dict('agreement_stimuli.csv', dataset)